# Simpson's Paradox #4 — HDI vs CO₂ Emissions
**Data sources:**
- CO₂ per capita: [Our World in Data / Global Carbon Project](https://github.com/owid/co2-data)
- Human Development Index (HDI): [UNDP HDR API](https://hdr.undp.org/data-center/documentation-and-downloads) + World Bank fallback

**The paradox:**  
Globally, higher HDI → higher CO₂ (more developed = more emissions).  
But *within* each HDI tier (Low / Medium / High / Very High), the relationship **flattens or reverses** —  
because the most-developed countries are also the ones aggressively decarbonising.

---
### Requirements
```bash
pip install pandas numpy matplotlib scipy requests
```

## 0 · Imports & Style

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import requests
import io
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "#0f0f1a",
    "axes.facecolor":   "#1a1a2e",
    "axes.edgecolor":   "#444466",
    "axes.labelcolor":  "#ccccee",
    "xtick.color":      "#aaaacc",
    "ytick.color":      "#aaaacc",
    "text.color":       "#e0e0f0",
    "grid.color":       "#2a2a44",
    "grid.alpha":       0.5,
    "font.family":      "DejaVu Sans",
    "font.size":        11,
})

# UNDP official HDI tier colours
TIER_COLORS = {
    "Low":       "#e05c5c",
    "Medium":    "#e0a040",
    "High":      "#6ab0e0",
    "Very High": "#66cc88",
}
TIER_ORDER = ["Low", "Medium", "High", "Very High"]

print("✅ Imports complete")

## 1 · Download CO₂ Data (OWID / Global Carbon Project)
Same source as Indicator #8 — we reuse the OWID CO₂ dataset.

In [ ]:
print("⬇  Downloading CO₂ data …")
CO2_URL = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
r = requests.get(CO2_URL, timeout=30)
owid = pd.read_csv(io.StringIO(r.text), low_memory=False)
owid = owid[owid["iso_code"].notna() & ~owid["iso_code"].str.startswith("OWID")]

WANT = ["country", "iso_code", "year", "co2_per_capita", "gdp", "population"]
AVAIL = [c for c in WANT if c in owid.columns]
co2_df = (owid[AVAIL]
          .query("2000 <= year <= 2022")
          .dropna(subset=["co2_per_capita"])
          .reset_index(drop=True))

print(f"   CO₂ rows: {len(co2_df):,}  |  countries: {co2_df['iso_code'].nunique()}")
print("✅ CO₂ download complete")

## 2 · Download HDI Data
**Primary source:** UNDP HDR bulk CSV (`hdr_composite_indices_complete_time_series.csv`)  
**Fallback:** World Bank API — indicator `HD.HCI.HLOSE` is not HDI itself, so we use  
a second fallback via the OWID COVID dataset which carries a static HDI per country.

The strategy below tries each source in order and stops at the first success.

In [ ]:
hdi_long = None

# ── Source 1: UNDP bulk time-series CSV ─────────────────────────────
print("⬇  Trying UNDP bulk CSV …")
UNDP_URL = (
    "https://hdr.undp.org/sites/default/files/2023-24_HDR/"
    "HDR23-24_Statistical_Annex_HDI_Trends_Table.csv"
)
try:
    r = requests.get(UNDP_URL, timeout=20)
    if r.status_code == 200 and "hdi" in r.text.lower():
        raw = pd.read_csv(io.StringIO(r.text), skiprows=1)
        # Melt year columns → long format
        id_cols = [c for c in raw.columns if not c.strip().isdigit()]
        yr_cols = [c for c in raw.columns if c.strip().isdigit()]
        if yr_cols:
            raw_long = raw.melt(id_vars=id_cols, value_vars=yr_cols,
                                var_name="year", value_name="hdi")
            raw_long["year"] = raw_long["year"].astype(int)
            raw_long = raw_long.dropna(subset=["hdi"])
            # Find country / iso columns
            ccol = next((c for c in id_cols if "country" in c.lower()), id_cols[0])
            icol = next((c for c in id_cols if "iso" in c.lower()), None)
            hdi_long = raw_long.rename(columns={ccol: "country"})
            if icol:
                hdi_long = hdi_long.rename(columns={icol: "iso_code"})
            hdi_long = hdi_long[["country"] + (["iso_code"] if icol else []) + ["year", "hdi"]]
            print(f"   UNDP bulk CSV: {len(hdi_long):,} rows")
except Exception as e:
    print(f"   UNDP bulk CSV failed: {e}")

# ── Source 2: UNDP composite indices complete time series ────────────
if hdi_long is None:
    print("⬇  Trying UNDP composite indices time-series …")
    UNDP_URL2 = (
        "https://hdr.undp.org/sites/default/files/2023-24_HDR/"
        "hdr_composite_indices_complete_time_series.csv"
    )
    try:
        r = requests.get(UNDP_URL2, timeout=20)
        if r.status_code == 200:
            raw = pd.read_csv(io.StringIO(r.text))
            print(f"   Columns: {list(raw.columns[:10])}")
            # HDI columns are named hdi_YYYY
            hdi_yr_cols = [c for c in raw.columns if c.startswith("hdi_") and c[4:].isdigit()]
            if hdi_yr_cols:
                id_cols = [c for c in raw.columns if not c.startswith("hdi_")]
                melted = raw.melt(id_vars=id_cols, value_vars=hdi_yr_cols,
                                  var_name="year_str", value_name="hdi")
                melted["year"] = melted["year_str"].str[4:].astype(int)
                melted = melted.dropna(subset=["hdi"])
                ccol = next((c for c in id_cols if "country" in c.lower()), id_cols[0])
                icol = next((c for c in id_cols if "iso" in c.lower()), None)
                hdi_long = melted.rename(columns={ccol: "country"})
                if icol:
                    hdi_long = hdi_long.rename(columns={icol: "iso_code"})
                keep = ["country"] + (["iso_code"] if icol else []) + ["year", "hdi"]
                hdi_long = hdi_long[keep]
                print(f"   UNDP time-series CSV: {len(hdi_long):,} rows")
    except Exception as e:
        print(f"   UNDP time-series failed: {e}")

# ── Source 3: OWID COVID dataset carries static HDI per country ──────
if hdi_long is None:
    print("⬇  Falling back to OWID COVID dataset (static HDI per country) …")
    COVID_URL = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/latest/owid-covid-latest.csv"
    try:
        r = requests.get(COVID_URL, timeout=30)
        cov = pd.read_csv(io.StringIO(r.text))
        cov = cov[cov["iso_code"].notna() & ~cov["iso_code"].str.startswith("OWID")]
        hdi_static = (cov[["location", "iso_code", "human_development_index"]]
                      .dropna(subset=["human_development_index"])
                      .rename(columns={"location": "country",
                                       "human_development_index": "hdi"}))
        # Replicate the static value for years 2000-2022 (good enough for cross-section)
        rows = []
        for _, row in hdi_static.iterrows():
            for yr in range(2000, 2023):
                rows.append({"country": row["country"],
                             "iso_code": row["iso_code"],
                             "year": yr,
                             "hdi": row["hdi"]})
        hdi_long = pd.DataFrame(rows)
        print(f"   OWID COVID fallback: {len(hdi_long):,} rows  "
              f"({hdi_static.shape[0]} countries × 23 years, static HDI)")
    except Exception as e:
        print(f"   OWID COVID fallback failed: {e}")

if hdi_long is None:
    raise RuntimeError("All HDI download sources failed. Check your internet connection.")

# Coerce and filter
hdi_long["hdi"]  = pd.to_numeric(hdi_long["hdi"], errors="coerce")
hdi_long["year"] = pd.to_numeric(hdi_long["year"], errors="coerce").astype("Int64")
hdi_long = hdi_long.dropna(subset=["hdi", "year"]).query("2000 <= year <= 2022")

print(f"\n✅ HDI data ready: {len(hdi_long):,} rows | "
      f"{hdi_long['country'].nunique()} countries")

## 3 · Merge & Assign HDI Tiers
UNDP official HDI tiers (2022 thresholds):
| Tier | HDI range |
|------|-----------|
| Very High | ≥ 0.800 |
| High | 0.700 – 0.799 |
| Medium | 0.550 – 0.699 |
| Low | < 0.550 |

In [ ]:
# Merge on iso_code if available, else on country name
if "iso_code" in hdi_long.columns:
    merged = pd.merge(co2_df, hdi_long[["iso_code", "year", "hdi"]],
                      on=["iso_code", "year"], how="inner")
else:
    merged = pd.merge(co2_df, hdi_long[["country", "year", "hdi"]],
                      on=["country", "year"], how="inner")

merged = merged.dropna(subset=["co2_per_capita", "hdi"])

def assign_tier(h):
    if h >= 0.800: return "Very High"
    if h >= 0.700: return "High"
    if h >= 0.550: return "Medium"
    return "Low"

merged["hdi_tier"] = merged["hdi"].apply(assign_tier)

# Focus year: 2019 (pre-COVID, richest data)
FOCUS_YEAR = 2019
df = merged[merged["year"] == FOCUS_YEAR].copy()

print(f"✅ Merged dataset: {len(df)} countries for {FOCUS_YEAR}")
print()
print(df["hdi_tier"].value_counts().reindex(TIER_ORDER))

## 4 · Regression Helper

In [ ]:
def ols(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 5:
        return None
    return stats.linregress(x[mask], y[mask])

x_col = "hdi"
y_col = "co2_per_capita"

reg_all = ols(df[x_col].values, df[y_col].values)
print(f"Aggregate slope: {reg_all.slope:+.4f}  "
      f"R²={reg_all.rvalue**2:.3f}  p={reg_all.pvalue:.2e}")
print()
print("Within-tier slopes:")
for tier in TIER_ORDER:
    sub = df[df["hdi_tier"] == tier]
    reg = ols(sub[x_col].values, sub[y_col].values)
    if reg:
        print(f"  {tier:<12}  slope={reg.slope:+.4f}  "
              f"R²={reg.rvalue**2:.3f}  n={len(sub)}")

## 5 · Figure 1 — Main Simpson's Paradox Scatter
The **white line** is the aggregate fit (positive slope: higher HDI → more CO₂).  
The **dashed coloured lines** are within-tier fits — notice how high-development countries
show a flat or negative slope as green policy decouples growth from emissions.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

for tier in TIER_ORDER:
    sub = df[df["hdi_tier"] == tier]
    col = TIER_COLORS[tier]
    ax.scatter(sub[x_col], sub[y_col], color=col, alpha=0.65,
               s=65, label=tier, zorder=3, edgecolors="none")
    reg = ols(sub[x_col].values, sub[y_col].values)
    if reg:
        xs = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
        ax.plot(xs, reg.slope * xs + reg.intercept,
                color=col, linewidth=2.2, linestyle="--", alpha=0.9, zorder=4)

# Aggregate line
xs_all = np.linspace(df[x_col].min(), df[x_col].max(), 200)
ax.plot(xs_all, reg_all.slope * xs_all + reg_all.intercept,
        color="white", linewidth=2.5, linestyle="-", alpha=0.9,
        label=f"Aggregate (slope={reg_all.slope:.2f})", zorder=5)

# Label selected countries
LABEL_ISO = {"USA", "CHN", "NOR", "IND", "NGA", "BRA", "RUS", "DEU", "AUS", "ZAF"}
for _, row in df[df["iso_code"].isin(LABEL_ISO)].iterrows():
    ax.annotate(row["iso_code"],
                xy=(row[x_col], row[y_col]),
                xytext=(4, 3), textcoords="offset points",
                fontsize=7.5, color="#ddddff", alpha=0.85)

# Explanation box
textstr = (
    "Simpson's Paradox:\n"
    f"► Aggregate slope = {reg_all.slope:.2f}  →  higher HDI = more CO₂\n\n"
    "► Within each HDI tier (dashed lines):\n"
    "   slope ≈ 0 or negative for Very High tier.\n"
    "   The aggregate slope is driven by\n"
    "   group-composition, not causality:\n"
    "   rich countries dominate both ends."
)
props = dict(boxstyle="round,pad=0.6", facecolor="#0a0a1a", alpha=0.85, edgecolor="#555577")
ax.text(0.02, 0.97, textstr, transform=ax.transAxes, fontsize=9,
        verticalalignment="top", bbox=props, color="#ccccee")

ax.set_xlabel("Human Development Index (HDI)  [0–1]", fontsize=13)
ax.set_ylabel("CO₂ per Capita  (tonnes)", fontsize=13)
ax.set_title(
    f"Simpson's Paradox — HDI vs CO₂ Emissions per Capita · {FOCUS_YEAR}\n"
    f"UNDP / OWID · n = {len(df)} countries",
    fontsize=14, pad=14)
ax.legend(loc="upper left", bbox_to_anchor=(0.02, 0.60),
          framealpha=0.3, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("hdi_co2_fig1_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 6 · Figure 2 — Slope Comparison by HDI Tier

In [ ]:
groups   = ["Aggregate"] + TIER_ORDER
slopes   = []
colors   = []

reg = ols(df[x_col].values, df[y_col].values)
slopes.append(reg.slope); colors.append("white")

for tier in TIER_ORDER:
    sub = df[df["hdi_tier"] == tier]
    reg = ols(sub[x_col].values, sub[y_col].values)
    slopes.append(reg.slope if reg else 0)
    colors.append(TIER_COLORS[tier])

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(groups, slopes, color=colors,
               edgecolor="#333355", linewidth=0.8, height=0.55)
ax.axvline(0, color="white", linewidth=1.2, linestyle="--", alpha=0.6)

for bar, val in zip(bars, slopes):
    ax.text(val + (0.5 if val >= 0 else -0.5),
            bar.get_y() + bar.get_height() / 2,
            f"{val:+.2f}", va="center",
            ha="left" if val >= 0 else "right",
            color="white", fontsize=10)

ax.set_xlabel("OLS Slope  (CO₂ per capita per HDI unit)", fontsize=11)
ax.set_title(
    "Simpson's Paradox — Regression Slopes by HDI Tier\n"
    "Large positive aggregate slope collapses within tiers",
    fontsize=12, pad=10)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("hdi_co2_fig2_slopes.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 · Figure 3 — Faceted View by HDI Tier
One panel per tier. The **solid coloured line** is the within-tier OLS fit;  
the **dashed white line** is the aggregate slope for reference.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, tier in enumerate(TIER_ORDER):
    ax   = axes[i]
    sub  = df[df["hdi_tier"] == tier]
    col  = TIER_COLORS[tier]

    ax.scatter(sub[x_col], sub[y_col], color=col, alpha=0.7,
               s=55, edgecolors="none", zorder=3)

    reg = ols(sub[x_col].values, sub[y_col].values)
    if reg:
        xs = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
        ax.plot(xs, reg.slope * xs + reg.intercept, color=col,
                linewidth=2.5, zorder=4,
                label=f"Within-tier slope: {reg.slope:+.2f}")

    xs_a = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
    ax.plot(xs_a, reg_all.slope * xs_a + reg_all.intercept,
            color="white", linewidth=1.5, linestyle="--", alpha=0.5,
            label=f"Aggregate slope: {reg_all.slope:+.2f}")

    # Label a few notable countries
    for _, row in sub[sub["iso_code"].isin(LABEL_ISO)].iterrows():
        ax.annotate(row["iso_code"],
                    xy=(row[x_col], row[y_col]),
                    xytext=(4, 2), textcoords="offset points",
                    fontsize=7.5, color="#ddddff")

    ax.set_title(f"{tier} HDI", fontsize=11, color=col, pad=6)
    ax.set_xlabel("HDI", fontsize=9)
    ax.set_ylabel("CO₂ / capita (t)", fontsize=9)
    ax.legend(fontsize=7.5, framealpha=0.25, loc="upper left")
    ax.grid(True, alpha=0.25)

fig.suptitle(
    f"Simpson's Paradox — HDI vs CO₂, Faceted by Tier  ({FOCUS_YEAR})\n"
    "Dashed white = aggregate slope · Solid = within-tier slope",
    fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("hdi_co2_fig3_faceted.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 · Figure 4 — Decoupling Trend in Very High HDI Countries (2000–2022)
This is where the paradox becomes a *real-world story*:  
the most developed countries have been growing their HDI **while cutting CO₂** —  
the decoupling that makes the within-tier slope flatten or reverse.

In [ ]:
very_high = merged[merged["hdi_tier"] == "Very High"].copy()

# For each year, compute mean HDI and mean CO₂ across Very High countries
trend = (very_high.groupby("year")
         .agg(mean_hdi=("hdi", "mean"),
              mean_co2=("co2_per_capita", "mean"),
              n=("country", "count"))
         .reset_index()
         .query("n >= 20"))        # only years with good coverage

fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

l1, = ax1.plot(trend["year"], trend["mean_hdi"],
               color="#66cc88", linewidth=2.5, label="Mean HDI (Very High tier)")
ax1.fill_between(trend["year"], trend["mean_hdi"], alpha=0.1, color="#66cc88")

l2, = ax2.plot(trend["year"], trend["mean_co2"],
               color="#e05c5c", linewidth=2.5, linestyle="--",
               label="Mean CO₂/cap (Very High tier)")

ax1.set_xlabel("Year", fontsize=12)
ax1.set_ylabel("Mean HDI", fontsize=12, color="#66cc88")
ax2.set_ylabel("Mean CO₂ per Capita (tonnes)", fontsize=12, color="#e05c5c")
ax1.tick_params(axis="y", labelcolor="#66cc88")
ax2.tick_params(axis="y", labelcolor="#e05c5c")

ax1.set_title(
    "Decoupling in Very High HDI Countries — HDI Rising, CO₂ Falling\n"
    "This within-tier reversal drives Simpson's Paradox",
    fontsize=13, pad=12)

lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="center left", fontsize=9, framealpha=0.3)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(trend["year"].min(), trend["year"].max())
plt.tight_layout()
plt.savefig("hdi_co2_fig4_decoupling.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 · Summary Statistics

In [ ]:
print("=" * 65)
print("SUMMARY: Simpson's Paradox — HDI vs CO₂ per Capita")
print("=" * 65)
print(f"{'Group':<15} {'N':>4}  {'Slope':>8}  {'R²':>6}  {'p-value':>12}")
print("-" * 65)

r = ols(df[x_col].values, df[y_col].values)
print(f"{'Aggregate':<15} {len(df):>4}  {r.slope:>8.2f}  {r.rvalue**2:>6.3f}  {r.pvalue:>12.2e}")

for tier in TIER_ORDER:
    sub = df[df["hdi_tier"] == tier]
    r = ols(sub[x_col].values, sub[y_col].values)
    if r:
        print(f"{tier:<15} {len(sub):>4}  {r.slope:>8.2f}  "
              f"{r.rvalue**2:>6.3f}  {r.pvalue:>12.2e}")

print("=" * 65)
print()
print("Key insight:")
print("  The aggregate slope is strongly positive (HDI ↑ → CO₂ ↑).")
print("  Within the Very High tier, the slope flattens or reverses,")
print("  because wealthy nations are decoupling growth from emissions.")
print()
print("Data sources:")
print("  CO₂:  https://github.com/owid/co2-data")
print("  HDI:  https://hdr.undp.org/data-center/documentation-and-downloads")
print()
print("✅ All figures saved.")
